# Confirm the ramp win: W-robustness + 10-fold gate

`smoothed_tau.ipynb`: ramp label +0.0095, t 4.52, 5/5. Before shipping / stacking, two checks:
1. **W-robustness** — is +0.0095 a knife-edge on W=15, or stable across the ramp width? Sweep
   W in {8, 15, 25, 40} at 5 folds. A real detection-lag effect should hold over a range.
2. **10-fold gate** — mandatory after step-sampling collapsed 5-fold t2.12 -> 10-fold t0.03.
   Re-run base + best-W ramp on 10 folds. t 4.52 with all-positive folds should survive, but verify.

Believe/ship only if the 10-fold paired delta stays t >= 3.

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled = cache['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
idx = pd.read_parquet(os.path.join(TOOLS, '..', '..', '..', 'y_train_index.parquet'))
tau_row = idx['tau_index'].reindex(np.unique(sid)).reindex(sid).to_numpy()
since = step - tau_row
is_break = tau_row >= 0

def ramp_target(W):
    r = np.clip(since / (2.0 * W), 0.0, 1.0)
    r[~is_break] = 0.0
    r[is_break & (since < 0)] = 0.0
    return pd.Series(r, index=index)

CLS = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
           random_seed=0, verbose=0, thread_count=-1)
REG = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='RMSE',
           random_seed=0, verbose=0, thread_count=-1)

def fp_base(train, val):
    m = CatBoostClassifier(**CLS); m.fit(train.X, train.y, sample_weight=train.w)
    return m.predict_proba(val.X)[:, 1]

def fp_ramp(W):
    tgt = ramp_target(W)
    def _f(train, val):
        m = CatBoostRegressor(**REG)
        m.fit(train.X, tgt.loc[train.index].to_numpy(), sample_weight=train.w)
        return m.predict(val.X)
    return _f

In [2]:
# --- 1. W-robustness at 5 folds ---
folds5 = CV.series_folds(sid, n_splits=5, seed=0)
res5 = {}
res5['base'] = CV.run_cv(X, y, sid, index, fp_base, folds=folds5,
                         train_row_mask=sampled, row_step=step, verbose=False)
print('base', res5['base'].summary('base'), flush=True)
for W in [8, 15, 25, 40]:
    t = time.time()
    r = CV.run_cv(X, y, sid, index, fp_ramp(W), folds=folds5,
                  train_row_mask=sampled, row_step=step, verbose=False)
    res5[f'ramp_W{W}'] = r
    d = CV.paired_compare(r, res5['base'], f'W{W}', 'base').split(chr(10))[0]
    print(f'ramp_W{W:<2d} {r.mean:.4f} | {d} | {time.time()-t:.0f}s', flush=True)
json.dump({k: v.fold_scores.tolist() for k, v in res5.items()},
          open('confirm_Wsweep.json', 'w'), indent=2)

base base               mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232


ramp_W8  0.6071 | W8 - base: +0.0094 +/- 0.0046 (sem 0.0021, t +4.56, wins 5/5) -> consistent | 210s


ramp_W15 0.6073 | W15 - base: +0.0095 +/- 0.0047 (sem 0.0021, t +4.52, wins 5/5) -> consistent | 210s


ramp_W25 0.6073 | W25 - base: +0.0096 +/- 0.0040 (sem 0.0018, t +5.38, wins 5/5) -> consistent | 216s


ramp_W40 0.6089 | W40 - base: +0.0112 +/- 0.0051 (sem 0.0023, t +4.87, wins 5/5) -> consistent | 198s


In [3]:
# --- 2. 10-fold gate on the best (or default W=15) ---
bestW = max([8, 15, 25, 40], key=lambda W: res5[f'ramp_W{W}'].mean)
print(f'best 5-fold W = {bestW}; confirming at 10 folds\n')
folds10 = CV.series_folds(sid, n_splits=10, seed=0)
b10 = CV.run_cv(X, y, sid, index, fp_base, folds=folds10,
                train_row_mask=sampled, row_step=step, verbose=False)
r10 = CV.run_cv(X, y, sid, index, fp_ramp(bestW), folds=folds10,
                train_row_mask=sampled, row_step=step, verbose=False)
print('base  10f', b10.summary('base'))
print('ramp  10f', r10.summary(f'ramp_W{bestW}'))
print('\n=== 10-FOLD GATE ===')
print(CV.paired_compare(r10, b10, f'ramp_W{bestW}', 'base'))
json.dump({'base10': b10.fold_scores.tolist(), f'ramp10_W{bestW}': r10.fold_scores.tolist()},
          open('confirm_10fold.json', 'w'), indent=2)

best 5-fold W = 40; confirming at 10 folds



base  10f base               mean 0.6043 +/- 0.0154 (sem 0.0049) | OOF 0.6033 | folds: 0.6040  0.5774  0.5980  0.6141  0.5887  0.5973  0.6144  0.5998  0.6223  0.6272
ramp  10f ramp_W40           mean 0.6119 +/- 0.0158 (sem 0.0050) | OOF 0.6111 | folds: 0.6175  0.5818  0.5978  0.6149  0.6035  0.6018  0.6190  0.6178  0.6327  0.6322

=== 10-FOLD GATE ===
ramp_W40 - base: +0.0076 +/- 0.0062 (sem 0.0020, t +3.87, wins 9/10) -> likely
  per-fold deltas: +0.0135  +0.0043  -0.0001  +0.0008  +0.0148  +0.0046  +0.0047  +0.0180  +0.0104  +0.0049
